# 1. A first transport calculation

**Learning goals.** By the end of this tutorial you will be able to:

- describe a single electronic level coupled to two reservoirs;
- translate the model parameters into a `qmeq.Builder` calculation;
- interpret occupations and the sign of the particle current; and
- check normalization, equilibrium, and current conservation.

Only elementary quantum mechanics, the Fermi function, and basic Python are assumed. We use natural units $\hbar=k_\mathrm{B}=|e|=1$ throughout.

## The physical question

Consider one spinless level of energy $\varepsilon$ between a left (L) and right (R) electron reservoir. The dot Hamiltonian is

$$H_\mathrm{dot}=\varepsilon d^\dagger d.$$

The dot has two many-body states: empty, $|0\rangle$, and occupied, $|1\rangle=d^\dagger|0\rangle$. Reservoir $\alpha$ has chemical potential $\mu_\alpha$, temperature $T_\alpha$, and coupling strength $\Gamma_\alpha$.

An electron can enter from reservoir $\alpha$ at rate $\Gamma_\alpha f_\alpha(\varepsilon)$ and leave at rate $\Gamma_\alpha[1-f_\alpha(\varepsilon)]$, where

$$f_\alpha(E)=\frac{1}{\exp[(E-\mu_\alpha)/T_\alpha]+1}.$$

**Prediction before calculating.** We choose $\mu_L>\varepsilon>\mu_R$. The level should be filled mainly from the left and emptied mainly into the right. QmeQ defines positive particle current as flowing *from a lead into the dot*, so we expect $I_L>0$, $I_R<0$, and $I_L+I_R=0$ in the stationary state.

In [ ]:
import numpy as np
import qmeq

print(qmeq.get_backend_status())

The numerical values below are measured relative to one arbitrary energy scale—for example, meV. Because $\hbar=k_\mathrm{B}=1$, temperatures, rates, voltages, and energies use that same scale.

In [ ]:
# Dot and reservoir parameters
epsilon = 0.0
mu_left, mu_right = 0.5, -0.5
temperature = 0.2
gamma_left, gamma_right = 0.2, 0.1
bandwidth = 20.0

# QmeQ uses density-of-states-weighted tunnelling amplitudes,
# Gamma = 2*pi*|t|^2.
t_left = np.sqrt(gamma_left / (2 * np.pi))
t_right = np.sqrt(gamma_right / (2 * np.pi))

## Translating the model into QmeQ

Indices start at zero. There is one single-particle state (`nsingle=1`) and two lead channels (`nleads=2`). Dictionary keys name matrix elements: `(0, 0)` in `hsingle` is the level energy, while `(lead, level)` in `tleads` identifies a tunnelling amplitude.

In [ ]:
system = qmeq.Builder(
    nsingle=1,
    hsingle={(0, 0): epsilon},
    coulomb={},
    nleads=2,
    tleads={(0, 0): t_left, (1, 0): t_right},
    mulst={0: mu_left, 1: mu_right},
    tlst={0: temperature, 1: temperature},
    dband=bandwidth,
    kerntype="Pauli",
)

system.solve()

For a Pauli master equation, `phi0` contains the stationary probabilities of the many-body states. In this simple model the entries are the probabilities $P_0$ and $P_1$ of the empty and occupied states.

In [ ]:
p_empty, p_occupied = system.phi0

print(f"P(empty)    = {p_empty:.6f}")
print(f"P(occupied) = {p_occupied:.6f}")
print("particle currents [L, R] =", system.current)

The signs agree with our prediction. `system.current[0]` is positive because particles enter from L; `system.current[1]` is negative because they leave into R. Electrical current additionally contains the carrier charge, whose sign must be restored when converting from particle current.

## Compare with the two-state rate equation

For this model the total rates into and out of the dot are

$$W_\mathrm{in}=\sum_\alpha\Gamma_\alpha f_\alpha(\varepsilon),\qquad
W_\mathrm{out}=\sum_\alpha\Gamma_\alpha[1-f_\alpha(\varepsilon)].$$

Stationarity and normalization give $P_1=W_\mathrm{in}/(W_\mathrm{in}+W_\mathrm{out})$ and $P_0=1-P_1$. This provides an analytical check of the numerical solver.

In [ ]:
def fermi(energy, chemical_potential, temp):
    return 1.0 / (np.exp((energy - chemical_potential) / temp) + 1.0)

f_left = fermi(epsilon, mu_left, temperature)
f_right = fermi(epsilon, mu_right, temperature)
w_in = gamma_left * f_left + gamma_right * f_right
w_out = gamma_left * (1 - f_left) + gamma_right * (1 - f_right)
p1_rate_equation = w_in / (w_in + w_out)
i_left_rate_equation = gamma_left * (
    f_left * (1 - p1_rate_equation) - (1 - f_left) * p1_rate_equation
)

print(f"analytical P(occupied) = {p1_rate_equation:.6f}")
print(f"analytical left current = {i_left_rate_equation:.6f}")

assert np.isclose(p_occupied, p1_rate_equation)
assert np.isclose(system.current[0], i_left_rate_equation)

## Three checks to perform in every stationary calculation

In [ ]:
# 1. Probabilities are normalized and non-negative.
assert np.isclose(np.sum(system.phi0), 1.0)
assert np.all(system.phi0 >= -1e-12)

# 2. Stationary particle current is conserved.
assert np.isclose(np.sum(system.current), 0.0, atol=1e-12)

# 3. Equal reservoirs produce no current.
system.change(mulst={0: 0.0, 1: 0.0})
system.solve(qdq=False)
assert np.allclose(system.current, 0.0, atol=1e-12)

print("normalization, positivity, conservation, and equilibrium checks passed")

## What this approximation does—and does not—describe

The Pauli equation follows only the probabilities of energy eigenstates. It is appropriate here when tunnelling is weak and coherences between different dot states are irrelevant. It does not generally capture coherent superpositions, level broadening, or cotunnelling. Later tutorials introduce those effects without treating one approximation as universally best.

## Exercises

1. Set `epsilon = 1.0` and predict how the occupation and current change before running the notebook. Both should become small because the level lies above both chemical potentials.
2. Exchange `mu_left` and `mu_right`. The current signs should reverse.
3. Set `gamma_left = gamma_right`. At the symmetric bias used above, symmetry implies $P_1=1/2$.